# Métodos Matemáticos Computacionales para Ciencia de Datos
## Laboratorio 04. Eigenpares, QR, escala de datos y análisis de componentes principales

**Nombre:** ______________________________________  
**Número de cuenta:** _____________________________  
**Fecha:** _______________________________________

**Valor total:** 68 puntos

## Flujo general del laboratorio

<p align="center">
  <img src="images/workflow.svg" width="1050">
</p>

El laboratorio conecta algoritmos espectrales, bases ortonormales, estructura computacional de matrices y reducción de dimensión.

# Instrucciones generales

Este laboratorio contiene **9 ejercicios** y tiene un valor total de **68 puntos**.

1. No agregue, elimine ni reordene celdas.
2. No modifique las celdas de sólo lectura.
3. Modifique únicamente las celdas que contienen:
   ```python
   # ESCRIBA SU CÓDIGO AQUÍ
   raise NotImplementedError()
   ```
4. Ejecute las celdas en orden.
5. Los `assert` visibles verifican propiedades básicas. La evaluación completa incluye pruebas ocultas.
6. Antes de entregar, reinicie el kernel y ejecute todo el notebook.
7. No use valores escritos manualmente para satisfacer los tests: implemente el método solicitado.

# Objetivos

Al finalizar el laboratorio podrá:

- implementar el **método de potencias** y estimar el eigenvalor dominante mediante el cociente de Rayleigh;
- construir una **factorización QR** mediante Gram–Schmidt modificado;
- relacionar dimensiones, número de elementos no cero y almacenamiento con la escala de un problema de datos;
- implementar productos matriz-vector aprovechando una representación dispersa simple;
- implementar **ACP/PCA** a partir de la matriz de covarianza;
- calcular varianza explicada, proyectar datos y reconstruir aproximaciones de dimensión reducida;
- aplicar ACP a un caso de estudio pequeño y seleccionar el número de componentes mediante un criterio cuantitativo.

In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)
TOL = 1e-10

# Ejercicio 1. Método de potencias — 8 puntos

<span style="color:red">
Este método presupone condiciones apropiadas para converger al eigenpar dominante, por ejemplo un eigenvalor dominante en magnitud y una componente inicial no nula en su dirección. Esto se discutió en clases.
</span>

Implemente `metodo_potencias(A, v0, max_iter=1000, tol=1e-10)`.

En cada iteración:

$$
w^{(k+1)} = A v^{(k)}, \qquad
v^{(k+1)} = \frac{w^{(k+1)}}{\|w^{(k+1)}\|_2}.
$$

Use el **cociente de Rayleigh**

$$
\lambda^{(k)}=
\frac{(v^{(k)})^T A v^{(k)}}{(v^{(k)})^T v^{(k)}}
$$

para estimar el eigenvalor. Detenga el proceso cuando

$$
\|v^{(k+1)}-v^{(k)}\|_2 < \text{tol}
$$

o cuando se alcance `max_iter`.

La función debe devolver `(lambda_dom, v_dom, n_iter)`.

Use este código guía

```python
def metodo_potencias(A, v0, max_iter=1000, tol=1e-10):
    A = np.asarray(_____, dtype=float)
    v = np.asarray(v0, dtype=float).reshape(-1)

    if A.ndim != 2 or _____.__________[0] != A.__________[1]:
        raise ValueError("A debe ser cuadrada.")
    if v.size != _____.__________[0]:
        raise ValueError("Dimensiones incompatibles.")
    nv = np.linalg.norm(____________________)
    if nv == 0:
        raise ValueError("v0 no puede ser el vector cero.")
    v = v / nv
    lam = float(____________________ @ (A @ ____________________))
    
    for k in range(1, max_iter + 1):
        w = A @ ____________________
        nw = np.linalg.norm(w)
        if nw == 0:
            raise ValueError("La iteración produjo el vector cero.")
        v_new = w / nw
        # corregir ambigüedad de signo para el criterio de convergencia
        if np.dot(v_new, ____________________) < 0:
            v_new = -v_new
        lam = float(v_new @ (A @ v_new) / (v_new @ v_new))
        if np.linalg.norm(v_new - v) < tol:
            return lam, v_new, k
        v = v_new
    return lam, ____________________, max_iter

lambda_1, v_1, niter_1 = metodo_potencias(A1, v0_1)
```

In [ ]:
A1 = np.array([
    [4., 1., 0.],
    [1., 3., 0.],
    [0., 0., 1.]
])
v0_1 = np.array([1., 1., 1.])

In [ ]:
# ESCRIBA SU CÓDIGO AQUÍ
raise NotImplementedError()

In [ ]:
# Puedes opcionalmente borrar esta celda. Contiene las pruebas ocultas.


# Ejercicio 2. Convergencia y cociente de Rayleigh — 8 puntos

Implemente `historial_potencias(A, v0, n_iter)` para ejecutar exactamente `n_iter` iteraciones del método de potencias.

Devuelva dos arreglos:

- `lambdas`: estimaciones del eigenvalor mediante el cociente de Rayleigh;
- `residuos`: $\|Av^{(k)}-\lambda^{(k)}v^{(k)}\|_2$.

No incluya el estado inicial en los arreglos.

Use el siguiente código guía:

```python
def historial_potencias(__________, v0, n_iter):
    A = np.asarray(__________, dtype=float)
    v = np.asarray(v0, dtype=float).reshape(-1)
    v = ____________________ / np.linalg.norm(v)
    lambdas = []
    residuos = []
    for _ in range(n_iter):
        w = __________ @ ____________________
        v = w / np.linalg.norm(w)
        lam = float(v @ (__________ @ ____________________) / (____________________ @ ____________________))
        r = float(np.linalg.norm(__________ @ ____________________ - lam * ____________________))
        lambdas.append(lam)
        residuos.append(r)
    return np.array(lambdas), np.array(residuos)

lambdas_2, residuos_2 = historial_potencias(A2, v0_2, 8)
```

In [ ]:
A2 = np.array([
    [5., 1.],
    [1., 2.]
])
v0_2 = np.array([1., 1.])

In [ ]:
# ESCRIBA SU CÓDIGO AQUÍ
raise NotImplementedError()

In [ ]:
# Puedes opcionalmente borrar esta celda. Contiene las pruebas ocultas.


# Ejercicio 3. Factorización QR con Gram–Schmidt modificado — 8 puntos

Implemente `qr_mgs(A)` para una matriz $A\in\mathbb{R}^{m\times n}$, $m\ge n$, con columnas linealmente independientes.

Debe construir una factorización QR reducida

$$
A=QR,\qquad Q^TQ=I,
$$

usando **Gram–Schmidt modificado**. No use `np.linalg.qr` dentro de la función.

Use el siguiente código guía:

```python
def qr_mgs(A):
    A = np.__________________(A, dtype=float)
    m, n = A.shape
    if m < n:
        raise ValueError("Se requiere m >= n.")
    Q = np.zeros((______________, ____________________________))
    R = np.zeros((____________________________, ____________________________))
    V = A.copy()
    for i in range(____________________________):
        R[i, i] = np.linalg.norm(____________[:, i])
        if R[i, i] < 1e-14:
            raise ValueError("Las columnas no son linealmente independientes.")
        Q[:, i] = ____________[:, i] / R[i, i]
        for j in range(i + 1, n):
            R[i, j] = Q[:, i] @ ____________[:, j]
            V[:, j] = ____________[:, j] - R[i, j] * Q[:, i]
    return Q, R

Q3, R3 = qr_mgs(A3)
```

In [ ]:
A3 = np.array([
    [1., 1., 0.],
    [1., 0., 1.],
    [0., 1., 1.],
    [1., 1., 1.]
])

In [ ]:
# ESCRIBA SU CÓDIGO AQUÍ
raise NotImplementedError()

In [ ]:
# Puedes opcionalmente borrar esta celda. Contiene las pruebas ocultas.


# Ejercicio 4. Mínimos cuadrados mediante QR — 6 puntos

Use su función `qr_mgs` para implementar `minimos_cuadrados_qr(A, b)`.

Para $A=QR$, resuelva

$$
R\hat{x}=Q^Tb.
$$

Puede usar `np.linalg.solve` para el sistema triangular final, pero no `np.linalg.lstsq` dentro de la función.

Use el siguiente código guía:

```python
def minimos_cuadrados_qr(A, b):
    ____________ = np.asarray(____________, dtype=float)
    __________________ = np.asarray(b, dtype=float)
    Q, R = qr_mgs(____________)
    return np.linalg.solve(R, Q.T @ __________________)

x4 = minimos_cuadrados_qr(________________________, _________)
residuo_4 = np.linalg.norm(________________________ @ x4 - _________)

```

In [ ]:
A4 = np.column_stack((np.ones(5), np.arange(5, dtype=float)))
b4 = np.array([1.0, 1.9, 3.2, 3.9, 5.1])

In [ ]:
# ESCRIBA SU CÓDIGO AQUÍ
raise NotImplementedError()

In [ ]:
# Puedes opcionalmente borrar esta celda. Contiene las pruebas ocultas.


# Ejercicio 5. ¿Qué tan grande y dispersa es una matriz? — 6 puntos

Implemente `metricas_matriz(X)` y devuelva:

`(n, p, nnz, densidad, bytes_denso)`

donde

$$
\operatorname{densidad}(X)=\frac{\operatorname{nnz}(X)}{np}.
$$

`bytes_denso` debe estimar el almacenamiento denso usando `X.nbytes`. El atributo `X.nbytes` calcula el total de memoria en bytes que ocupan los elementos de una estructura de datos [[Ver documentación de Numpy](https://numpy.org/devdocs/reference/generated/numpy.ndarray.nbytes.html)]. 

<span style="padding:10px;background-color:#ff0000">En este laboratorio, **dispersa** (sparse) significa que una gran fracción de las entradas es cero. No confunda dispersidad con varianza estadística.</span>

Use el siguiente código guía:

```python
def metricas_matriz(X):
    X = np.asarray(________________)
    n, p = ________________.shape
    nnz = int(np.count_nonzero(X))
    densidad = ________________________________ / (n * p)
    bytes_denso = int(X.nbytes)
    return n, p, ________________________________, densidad, bytes_denso

n5, p5, nnz5, densidad5, bytes5 = metricas_matriz(X5)
```

In [ ]:
X5 = np.zeros((100, 80), dtype=float)
X5[::10, ::8] = 1.0

In [ ]:
# ESCRIBA SU CÓDIGO AQUÍ
raise NotImplementedError()

In [ ]:
# Puedes opcionalmente borrar esta celda. Contiene las pruebas ocultas.


# Ejercicio 6. Producto matriz-vector con datos dispersos — 8 puntos

Una matriz dispersa puede representarse guardando sólo las entradas no cero mediante tres arreglos:

- `rows[k]`: fila del elemento;
- `cols[k]`: columna del elemento;
- `values[k]`: valor.

Implemente

`matvec_coo(rows, cols, values, shape, x)`

sin construir la matriz densa. Debe devolver $y=Ax$.

Este ejercicio ilustra que, si sólo se recorren los elementos no cero, el trabajo puede relacionarse con $\operatorname{nnz}(A)$ en vez de $np$.

Use el siguiente código guía:
```python
def matvec_coo(rows, cols, values, shape, x):
    rows = np.asarray(rows, dtype=int)
    cols = np.asarray(cols, dtype=int)
    values = np.asarray(__________________, dtype=float)
    x = np.asarray(x, dtype=float)
    m, n = shape
    if x.size != n:
        raise ValueError("Dimensiones incompatibles.")
    y = np.zeros(m, dtype=float)
    for i, j, aij in zip(rows, cols, __________________):
        y[i] += aij * x[j]
    return _________

y6 = matvec_coo(rows6, cols6, vals6, shape6, x6)
```

In [ ]:
rows6 = np.array([0, 0, 1, 3, 4])
cols6 = np.array([0, 3, 1, 2, 4])
vals6 = np.array([2., -1., 3., 4., 5.])
shape6 = (5, 5)
x6 = np.array([1., 2., 3., 4., 5.])

In [ ]:
# ESCRIBA SU CÓDIGO AQUÍ
raise NotImplementedError()

In [ ]:
# Puedes opcionalmente borrar esta celda. Contiene las pruebas ocultas.

# Ejercicio 7. ACP desde la matriz de covarianza — 8 puntos

Implemente `ajustar_pca(X)` sin usar `sklearn`.

La función debe:

1. calcular la media por columna;
2. centrar los datos;
3. construir
   $$
   S=\frac{1}{n-1}X_c^TX_c;
   $$
4. usar `np.linalg.eigh(S)`;
5. ordenar eigenvalores y eigenvectores de mayor a menor;
6. devolver `(media, eigenvalores, componentes)`.

Las **columnas** de `componentes` deben ser los eigenvectores ordenados.

Use el siguiente código guía:

```python
def ajustar_pca(X):
    X = np.asarray(X, dtype=float)
    media = X.________________(axis=0)
    Xc = X - ______________
    S = (__________________________.T @ __________________________) / (X.shape[0] - 1)
    vals, vecs = np.linalg.eigh(S)
    idx = np.argsort(vals)[::-1]
    vals = vals[idx]
    vecs = vecs[:, idx]
    return ______________, vals, vecs

media7, eigvals7, componentes7 = ajustar_pca(X7)
```

In [ ]:
X7 = np.array([
    [2., 1.],
    [3., 2.],
    [4., 3.],
    [5., 4.]
])

In [ ]:
# ESCRIBA SU CÓDIGO AQUÍ
raise NotImplementedError()

In [ ]:
# Puedes opcionalmente borrar esta celda. Contiene las pruebas ocultas.


# Ejercicio 8. Proyección, varianza explicada y reconstrucción — 8 puntos

Implemente:

`transformar_pca(X, media, componentes, k)`

y

`reconstruir_pca(Z, media, componentes, k)`.

Use

$$
Z=X_cV_k,\qquad
\hat X=ZV_k^T+\mu.
$$

Además calcule `varianza_explicada_8` para cada componente:

$$
\mathrm{EVR}_i=\frac{\lambda_i}{\sum_j\lambda_j}.
$$

Use el modelo ACP calculado en el ejercicio anterior.

Use el siguiente código guía:

```python
def transformar_pca(X, media, componentes, k):
    X = np.______________________(X, dtype=float)
    Vk = ____________________________________________[:, :k]
    return (X - media) @ Vk

def reconstruir_pca(Z, media, ____________________________________________, k):
    Vk = ____________________________________________[:, :k]
    return np.asarray(Z, dtype=float) @ Vk.T + media

Z8 = transformar_pca(X7, media7, componentes7, 1)
Xrec8 = reconstruir_pca(Z8, media7, componentes7, 1)
varianza_explicada_8 = eigvals7 / eigvals7.sum()
```


In [ ]:
# ESCRIBA SU CÓDIGO AQUÍ
raise NotImplementedError()

In [ ]:
# Puedes opcionalmente borrar esta celda. Contiene las pruebas ocultas.


# Ejercicio 9. Caso de estudio: selección automática de componentes — 8 puntos

En un caso de Ciencia de Datos queremos conservar el menor número de componentes cuya **varianza explicada acumulada** alcance un umbral $\tau$.

Implemente:

`seleccionar_k(eigenvalores, umbral=0.90)`

que devuelva el menor entero $k$ tal que

$$
\frac{\sum\limits_{i=1}^{k}\lambda_i}
{\sum\limits_{j=1}^{p}\lambda_j}
\ge \tau.
$$

Después aplique ACP al conjunto `X9`, seleccione $k$ para un umbral de 90% y construya `Z9`.

Use el siguiente código guía:

```python
def seleccionar_k(eigenvalores, umbral=0.90):
    vals = np.asarray(eigenvalores, dtype=float)
    if vals.____________ != 1 or vals.size == 0:
        raise ValueError("eigenvalores debe ser un vector no vacío.")
    if not (0 < ________________________ <= 1):
        raise ValueError("________________________ debe pertenecer a (0,1].")
    acumulada = np.cumsum(vals) / np.sum(vals)
    return int(np.searchsorted(acumulada, ________________________) + 1)

media9, eigvals9, componentes9 = ajustar_pca(X9_std)
k9 = seleccionar_k(eigvals9, 0.90)
Z9 = transformar_pca(X9_std, media9, componentes9, k9)
evr9 = ____________________________________ / ____________________________________.sum()
```

In [ ]:
X9 = np.array([
    [2.0,  20.,  1.0,  5.0],
    [2.5,  24.,  1.2,  5.4],
    [3.0,  31.,  1.5,  6.1],
    [3.5,  34.,  1.8,  6.5],
    [4.0,  41.,  2.0,  7.2],
    [4.5,  45.,  2.2,  7.6],
    [5.0,  52.,  2.6,  8.4],
    [5.5,  55.,  2.7,  8.8]
], dtype=float)

# Estandarizamos porque las variables están en escalas diferentes.
media9_raw = X9.mean(axis=0)
std9_raw = X9.std(axis=0, ddof=1)
X9_std = (X9 - media9_raw) / std9_raw

In [ ]:
# ESCRIBA SU CÓDIGO AQUÍ
raise NotImplementedError()

In [ ]:
# Puedes opcionalmente borrar esta celda. Contiene las pruebas ocultas.


# Resumen
En este laboratorio trabajó numéricamente con:

1. Método de potencias
2. Rayleigh
3. QR y GS
4. Mínimos cuadrados QR
5. Escala y dispersidad
6. ACP 

Antes de entregar:

1. reinicie el kernel;
2. ejecute todas las celdas;
3. verifique que no existan errores;
4. guarde el archivo con el nombre solicitado por el profesor.